<a href="https://colab.research.google.com/github/prithwis/GeoSim/blob/main/gsWorldDataStatic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

In [1]:
!pip -q install pymupdf
!wget -O gsLLM.py -q https://raw.githubusercontent.com/prithwis/GeoSim/refs/heads/main/gsUtils/gsLLM.py
!wget -O gsWorldFacts.py -q https://raw.githubusercontent.com/prithwis/GeoSim/refs/heads/main/gsUtils/gsWorldFacts.py                  # all

from gsLLM import *
from gsWorldFacts import *

from pathlib import Path
import time

In [2]:
import importlib
import gsLLM, gsWorldFacts
importlib.reload(gsLLM)
importlib.reload(gsWorldFacts)

<module 'gsWorldFacts' from '/content/gsWorldFacts.py'>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

WORLDFACTS = "/content/drive/MyDrive/Prithwis/GeoSim/Silicon/WorldFacts"

!ls -l "{WORLDFACTS}"

Mounted at /content/drive
total 16
drwx------ 2 root root 4096 Nov 19 07:16 Markdown
drwx------ 2 root root 4096 Nov 19 07:16 PDF_Original
drwx------ 2 root root 4096 Nov 21 04:46 PDF_test
-rw------- 1 root root 3421 Nov 19 07:34 Silicon_Stack_Source_Manifest.md


## Static World Data
URLs stored as PDF files that are converted to markdown files

In [4]:
# ------------------------------------------------
# Use OpenAI
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
model = "gpt-6-luna"
#model = "gpt-6.1-sol"
#model = "gpt-6-astra"

# ------------------------------------------------
# Use Claude
# ------------------------------------------------
#!pip install -q anthropic

#provider = "claude"
#model = "claude-haiku-5-5"
#model = "claude-sonnet-5-5"

# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF


In [5]:
PDF_DIR = f"{WORLDFACTS}/PDF_test"
MD_DIR  = f"{WORLDFACTS}/Markdown"

!ls -l "{PDF_DIR}"

total 12980
-rw------- 1 root root 5077727 Nov 21 04:48 Copy01-MappingTheSemiConductorValueChain.pdf
-rw------- 1 root root 8213160 Nov 21 04:48 Copy05-EnergyandAI.pdf


In [6]:
from IPython.display import display, Markdown
from pathlib import Path

manifest = Path(WORLDFACTS) / "Silicon_Stack_Source_Manifest.md"
#display(Markdown(manifest.read_text(encoding="utf-8")))

In [7]:
ROLE_DIR = Path(WORLDFACTS).parent / "Roles"
Role_PDFReader = ((ROLE_DIR / "RoleFile_PDFCleaner.txt").read_text(encoding="utf-8").strip())
#print(Role_PDFReader)

In [8]:

from pathlib import Path

pdf_files = sorted(Path(PDF_DIR).glob("*.pdf"))
md_dir = Path(MD_DIR)
md_dir.mkdir(parents=True, exist_ok=True)

print(f"Found {len(pdf_files)} PDF files\n")

md_files = []
failed_files = []

for i, source_PDF in enumerate(pdf_files, 1):

    print("=" * 70)
    print(f"[{i}/{len(pdf_files)}] Processing: {source_PDF.name}")

    try:
        md_path = GetWorldFile(
            client,
            Role_PDFReader,
            source_PDF,
            MD_DIR,
            model,
            provider
        )

        md_path = Path(md_path)

        if not md_path.exists():
            raise FileNotFoundError(
                f"Markdown output not created: {md_path}"
            )

        if md_path.stat().st_size == 0:
            raise ValueError("Generated Markdown is empty")

        md_files.append(md_path)

        print(f"SUCCESS: {md_path.name}")
        print(f"Markdown size: {md_path.stat().st_size:,} bytes")

    except Exception as e:
        failed_files.append((source_PDF.name, str(e)))
        print(f"FAILED: {source_PDF.name}")
        print(f"Reason: {e}")

print("\n" + "=" * 70)
print(f"Completed: {len(md_files)} of {len(pdf_files)} files")
print(f"Failed: {len(failed_files)}")

for name, reason in failed_files:
    print(f"  {name}: {reason}")


Found 2 PDF files

[1/2] Processing: Copy01-MappingTheSemiConductorValueChain.pdf
Reading: Copy01-MappingTheSemiConductorValueChain.pdf
Pages   : 59
Batches : 2
Cleaning 1/2 (149,796 characters)
OPENAI | gpt-6-luna | Effort: medium | In: 34,290 | Cached: 0 | Out: 31,217 | Reasoning: 391 | Time: 165.9s
Output: 143,060 characters
Cleaning 2/2 (347 characters)
OPENAI | gpt-6-luna | Effort: medium | In: 409 | Cached: 0 | Out: 166 | Reasoning: 105 | Time: 3.4s
Output: 237 characters
Created  : Copy01MappingTheSemi_261121.md
Markdown : 143,420 characters
SUCCESS: Copy01MappingTheSemi_261121.md
Markdown size: 143,876 bytes
[2/2] Processing: Copy05-EnergyandAI.pdf
Reading: Copy05-EnergyandAI.pdf
Pages   : 304
Batches : 6
Cleaning 1/6 (149,210 characters)
OPENAI | gpt-6-luna | Effort: medium | In: 32,275 | Cached: 0 | Out: 26,923 | Reasoning: 302 | Time: 149.6s
Output: 133,689 characters
Cleaning 2/6 (148,527 characters)
OPENAI | gpt-6-luna | Effort: medium | In: 32,832 | Cached: 0 | Out: 27,77

In [10]:
!ls -l "{MD_DIR}"

total 1411
-rw------- 1 root root 142706 Nov 19 08:08 01MappingTheSemiCond_261119.md
-rw------- 1 root root  82395 Nov 19 08:10 02ChipLandscape_261119.md
-rw------- 1 root root  10283 Nov 19 08:11 03BCGStrengtheningth_261119.md
-rw------- 1 root root  85960 Nov 19 08:13 04ShivakumarExportCo_261119.md
-rw------- 1 root root 223861 Nov 19 08:28 05EnergyandAI_261119.md
-rw------- 1 root root 143876 Nov 21 04:55 Copy01MappingTheSemi_261121.md
-rw------- 1 root root 753441 Nov 21 05:10 Copy05EnergyandAI_261121.md


In [12]:
#mdFile = Path(MD_DIR) / "Copy05EnergyandAI_261121.md"
#display(Markdown(mdFile.read_text(encoding="utf-8")))

In [13]:
for f in md_files:
    print(f.name)

Copy01MappingTheSemi_261121.md
Copy05EnergyandAI_261121.md
